In [ ]:
import gym, torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical
from tqdm.notebook import tqdm

env = gym.make('LunarLander-v2', render_mode="rgb_array")

## Overview

| Method            | Type          | Advantages                                  | Challenges                                         |
|-------------------|---------------|---------------------------------------------|----------------------------------------------------|
| Policy Gradient    | Policy-based  | Good for continuous action spaces           | High variance, can be unstable                     |
| DQN               | Value-based   | Simple, good for discrete actions           | Struggles with continuous actions, stability issues|
| A2C (Actor-Critic)| Hybrid        | Efficient, low variance                     | Complex, requires careful tuning                   |


## Policy Gradient

搭建簡單的 policy network。
我們預設模型的輸入是 8-dim 的 observation，輸出則是離散的四個動作之一：

In [ ]:
class PolicyGradientNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        '''
        Define the layers of the network
        1. First fully connected layer: takes an input of size 8 and outputs 16 neurons
        2. Second fully connected layer: takes input of size 16 and outputs 16 neurons
        3. Third fully connected layer: takes input of size 16 and outputs 4 neurons

        This could represent the number of possible actions in an environment (e.g., 4 actions)
        '''
        self.fc1 = nn.Linear(8, 16)
        self.fc2 = nn.Linear(16, 16)
        self.fc3 = nn.Linear(16, 4)

    def forward(self, state):
        '''
        Pass the input through the first layer and apply the tanh activation function
        Tanh squeezes the output to a range between -1 and 1
        '''
        hid = torch.tanh(self.fc1(state))
        hid = torch.tanh(self.fc2(hid))
        
        '''
        Pass the result through the third layer and apply the softmax function
        Softmax normalizes the output to represent probabilities (sums up to 1)
        dim=-1 means the softmax is applied across the last dimension
        ''' 
        return F.softmax(self.fc3(hid), dim=-1)


再來，搭建 agent，並搭配上方的 policy network 來採取行動。
這個 agent 能做到以下幾件事：
- `learn()`：從記下來的 log probabilities 及 rewards 來更新 policy network。
- `sample()`：從 environment 得到 observation 之後，利用 policy network 得出應該採取的行動。
而此函式除了回傳抽樣出來的 action，也會回傳此次抽樣的 log probabilities。

In [ ]:
class PolicyGradientAgent(nn.Module):
    def __init__(self, network):
        super().__init__()

        # Initialize the network (the neural network that will be used for decision making)
        self.network = network

        # Set up an optimizer (Stochastic Gradient Descent) to update the network's parameters
        self.optimizer = optim.SGD(self.network.parameters(), lr=0.001)
    
    
    def forward(self, state):
        # This simply passes the input (state) through the network
        return self.network(state)

    def learn(self, log_probs, rewards):
        '''
        Calculate the loss (the negative log probabilities multiplied by the rewards)
        The agent will learn by maximizing rewards, which means minimizing this negative loss
        '''
        loss = (-log_probs * rewards).sum() 
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
    
    # Define the sample method to select an action based on the network's output
    def sample(self, state):
        # Pass the state (input) through the network to get action probabilities
        action_prob = self.network(torch.FloatTensor(state))

        # Create a categorical distribution based on the action probabilities
        action_dist = Categorical(action_prob)

        # Sample an action from the distribution
        action = action_dist.sample()

        # Return the selected action and its log probability
        log_prob = action_dist.log_prob(action)

        # Return the selected action and its log probability
        return action.item(), log_prob



根據上述的 gradient-based network，初始化一位 agent，用於後續的訓練。

In [ ]:
network = PolicyGradientNetwork()
agent = PolicyGradientAgent(network)

## 訓練 Agent

現在我們開始訓練 agent。
透過讓 agent 和 environment 互動，我們記住每一組對應的 log probabilities 及 reward，並在成功登陸或者不幸墜毀後，回放這些「記憶」來訓練 policy network。

In [ ]:

agent.network.train() # Make sure the agent's network is in training mode

''' Define some hyperparameters '''
EPISODE_PER_BATCH = 5  # Number of episodes to collect before updating the agent
NUM_BATCH = 400       # Number of batches for updating the agent
GAMMA = 0.8           # Discount factor for future rewards (higher means future rewards are valued more)
acc_reward = 0         # Initialize the accumulated reward


'''Lists to store the average total and final rewards for monitoring progress'''
avg_total_rewards, avg_final_rewards = [], []

prg_bar = tqdm(range(NUM_BATCH))
for batch in prg_bar:

    '''Lists to store log probabilities, rewards, and other values for the current batch'''
    log_probs_episode, log_probs, episode_rewards, rewards = [], [], [], []
    total_rewards, final_rewards = [], []

    # Collect training data for a batch of episodes
    for episode in range(EPISODE_PER_BATCH):
        state, info = env.reset() # Reset the environment at the start of each episode and get the initial state
        total_reward, total_step = 0, 0  # Initialize total reward and step counter
        
        while True:
            # Get action and log probability (log(at|st))
            action, log_prob = agent.sample(state) 

            # Perform the action in the environment and get the next state and reward
            next_state, reward, done, truncate, info = env.step(action)

            # Store the log probability of the action taken
            log_probs_episode.append(log_prob) # [log(a1|s1), log(a2|s2), ...., log(at|st)]

            # Update the state, reward, and step counter
            total_reward += reward
            total_step += 1
            episode_rewards.append(reward)

            if done:
                final_rewards.append(reward)
                total_rewards.append(total_reward)
                break
        
        '''
        Adjust 'immediate reward' to 'accumulative decaying reward' with discount factor gamma = 0.99
        
        Current: 
            - action list     : a1, a2, a3, ...
            - imdeiate reward : r1, r2, r3, ...
        
        Accumulative Decaying reward
            - action list     : a1                         , a2                         ,  a3,
            - decaying reward : r1+0.99*r2+0.99^2*r3+......, r2+0.99*r3+0.99^2*r4+......,  r3+0.99*r4+0.99^2*r5+ ......
        '''
        decayed_rewards = []
        acc_reward = 0
        for r in reversed(episode_rewards):  # Iterate rewards in reverse order
            acc_reward = r + GAMMA * acc_reward  # Apply decay factor
            decayed_rewards.insert(0, acc_reward)  # Insert at the beginning

        rewards.extend(decayed_rewards)  # Extend the main rewards list with decayed rewards
        log_probs.extend(log_probs_episode)
        
    # 紀錄訓練過程
    avg_total_reward = sum(total_rewards) / len(total_rewards)
    avg_final_reward = sum(final_rewards) / len(final_rewards)
    avg_total_rewards.append(avg_total_reward)
    avg_final_rewards.append(avg_final_reward)
    prg_bar.set_description(f"Total: {avg_total_reward: 4.1f}, Final: {avg_final_reward: 4.1f}")

    # 更新網路
    rewards = (rewards - np.mean(rewards)) / (np.std(rewards) + 1e-9)  # 將 reward 正規標準化
    agent.learn(torch.stack(log_probs), torch.from_numpy(rewards))
    